In [ ]:
# ==============================================================================
# Nielsen's Heuristics Baseline + Optimized Preference Pipeline
#
# Experiment:
#   - Baseline  (Phase D-N): score test pairs using raw Nielsen's 10 heuristics
#   - Optimized (Phase D-R): score test pairs using user-adapted revised Nielsen's
#   - Compare accuracy of both, output revised heuristic text with diffs
#
# Flow:
#   Phase A  → per-pair feature extraction + per-pair Nielsen revision hints
#   Phase C  → aggregate revision hints → produce Revised Nielsen's (10 heuristics)
#   Phase D-N → evaluate test pairs with ORIGINAL Nielsen's
#   Phase D-R → evaluate test pairs with REVISED Nielsen's
#   Output   → accuracy comparison + revised heuristics JSON
# ==============================================================================
import os
import json
import base64
import time
import random
import re
import urllib.request
import urllib.error
import http.client
import concurrent.futures
from typing import List, Dict, Any, Optional
from pathlib import Path
import pandas as pd
from PIL import Image
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

# ==============================================================================
# CONFIGURATION
# ==============================================================================
ANNOTATOR_DIR   = "./annotator"
IMAGES_DIR      = "./screen"
RESULTS_DIR     = "./results_nielsen_experiment"
MODEL           = "gpt-5.4"
TRAIN_SIZE      = 30
TEST_SIZE       = 30       # Cap test set at this many rows (None = use all remaining)
SEED            = 7
MAX_WORKERS     = 7
PHASE_D_VOTE_RUNS          = 3
NON_DISCRIMINATING_MARGIN  = 1.0
TIE_MARGIN                 = 2.0

AZURE_OPENAI_TARGET_URI = os.getenv(
    "AZURE_OPENAI_TARGET_URI",
    f"https://wu-lab-east-us-2.openai.azure.com/openai/deployments/{MODEL}/chat/completions?api-version=2025-01-01-preview"
)
AZURE_OPENAI_API_KEY = "XXX"

# ==============================================================================
# NIELSEN'S 10 HEURISTICS (canonical baseline)
# Each has a weight of 1.0 — equal footing at baseline.
# ==============================================================================
NIELSENS_ORIGINAL = [
    {
        "id": "H1",
        "criterion": "visibility_of_system_status",
        "description": "The design clearly communicates current state, feedback, and progress to the user.",
        "weight": 1.0,
    },
    {
        "id": "H2",
        "criterion": "match_between_system_and_real_world",
        "description": "Language, icons, and concepts match the user's real-world mental models.",
        "weight": 1.0,
    },
    {
        "id": "H3",
        "criterion": "user_control_and_freedom",
        "description": "Users can easily undo, redo, or exit without being trapped.",
        "weight": 1.0,
    },
    {
        "id": "H4",
        "criterion": "consistency_and_standards",
        "description": "UI follows platform conventions; similar elements behave and look the same.",
        "weight": 1.0,
    },
    {
        "id": "H5",
        "criterion": "error_prevention",
        "description": "Design prevents mistakes through constraints, confirmations, or clear affordances.",
        "weight": 1.0,
    },
    {
        "id": "H6",
        "criterion": "recognition_over_recall",
        "description": "Options, actions, and information are visible rather than requiring memorisation.",
        "weight": 1.0,
    },
    {
        "id": "H7",
        "criterion": "flexibility_and_efficiency",
        "description": "Shortcuts and accelerators let expert users work faster without blocking novices.",
        "weight": 1.0,
    },
    {
        "id": "H8",
        "criterion": "aesthetic_and_minimalist_design",
        "description": "Only relevant information is shown; visual clutter is minimised.",
        "weight": 1.0,
    },
    {
        "id": "H9",
        "criterion": "help_users_recognise_diagnose_recover_from_errors",
        "description": "Error messages are plain-language, precise, and constructively suggest a solution.",
        "weight": 1.0,
    },
    {
        "id": "H10",
        "criterion": "help_and_documentation",
        "description": "Contextual help is easy to find, task-focused, and concise.",
        "weight": 1.0,
    },
]

# ==============================================================================
# HELPER FUNCTIONS  (unchanged from original)
# ==============================================================================
def set_seed(seed: int):
    random.seed(seed)

MAX_SIZE = (1024, 1024)

def encode_image(image_path: str) -> Optional[str]:
    try:
        img = Image.open(image_path).convert("RGB")
        img.thumbnail(MAX_SIZE, Image.Resampling.LANCZOS)
        import io
        buf = io.BytesIO()
        img.save(buf, format="JPEG", quality=85)
        return base64.b64encode(buf.getvalue()).decode("utf-8").replace("\n", "")
    except Exception as e:
        print(f"  [!] Error encoding image {image_path}: {e}")
        return None

def extract_json_from_text(text: str) -> dict:
    try:
        match = re.search(r'```(?:json)?\s*(.*?)\s*```', text, re.DOTALL)
        if match:
            return json.loads(match.group(1))
        return json.loads(text)
    except json.JSONDecodeError as e:
        print(f"  [!] JSON Parse Error. Raw text:\n{text}\n")
        raise e

def call_llm(
    prompt: str,
    image_a_b64: Optional[str] = None,
    image_b_b64: Optional[str] = None,
    max_retries: int = 5,
    temperature: float = 0.2,
) -> Dict[str, Any]:
    content = [{"type": "text", "text": prompt}]
    if image_a_b64 and image_b_b64:
        content.extend([
            {"type": "text", "text": "Image A:"},
            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{image_a_b64}"}},
            {"type": "text", "text": "Image B:"},
            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{image_b_b64}"}},
        ])
    payload = {
        "messages": [{"role": "user", "content": content}],
        "temperature": temperature,
        "max_completion_tokens": 1200,
    }
    headers = {
        "Content-Type": "application/json",
        "api-key": AZURE_OPENAI_API_KEY,
        "User-Agent": "Jupyter-LLM-Evaluator/1.0",
    }
    data = json.dumps(payload).encode("utf-8")
    for attempt in range(max_retries):
        try:
            req = urllib.request.Request(
                AZURE_OPENAI_TARGET_URI, data=data, headers=headers, method="POST"
            )
            with urllib.request.urlopen(req, timeout=30) as response:
                result = json.loads(response.read().decode("utf-8"))
                content_str = result["choices"][0]["message"]["content"]
                return extract_json_from_text(content_str)
        except urllib.error.HTTPError as e:
            try:
                body = e.read().decode("utf-8")
                print(f"  [!] HTTP {e.code} (Attempt {attempt+1}/{max_retries}): {body}")
            except Exception:
                print(f"  [!] HTTP {e.code} (Attempt {attempt+1}/{max_retries}): {e.reason}")
            if e.code == 400:
                raise
            if attempt < max_retries - 1:
                time.sleep(2 + (2 ** attempt))
            else:
                raise
        except (urllib.error.URLError, http.client.RemoteDisconnected, ConnectionResetError) as e:
            print(f"  [!] Connection Error (Attempt {attempt+1}/{max_retries}): {e}")
            if attempt < max_retries - 1:
                time.sleep(2 + (2 ** attempt))
            else:
                raise
        except Exception as e:
            print(f"  [!] API Error (Attempt {attempt+1}/{max_retries}): {e}")
            if attempt < max_retries - 1:
                time.sleep(2 + (2 ** attempt))
            else:
                raise
    return {}

# ==============================================================================
# DATA HELPERS  (unchanged)
# ==============================================================================
def get_winner_label(row: pd.Series) -> str:
    choice = str(row.get("final_choice", "")).strip()
    if choice in ("A < B", "A << B"):
        return "B"
    return "A"

def get_preference_strength(row: pd.Series) -> int:
    choice = str(row.get("final_choice", "")).strip()
    if ">>" in choice or "<<" in choice:
        return 2
    return 1

def load_annotator_data(csv_path: str) -> pd.DataFrame:
    return pd.read_csv(csv_path)

def sample_and_split(df: pd.DataFrame, seed: int) -> tuple[pd.DataFrame, pd.DataFrame]:
    strong = df[df["final_choice"].str.contains(r">>|<<", na=False)]
    weak   = df[~df["final_choice"].str.contains(r">>|<<", na=False)]
    n_strong = min(len(strong), round(TRAIN_SIZE * len(strong) / len(df)))
    n_weak   = TRAIN_SIZE - n_strong
    sampled_strong = strong.sample(n=n_strong, random_state=seed) if n_strong > 0 else strong.iloc[:0]
    sampled_weak   = weak.sample(n=n_weak,     random_state=seed) if n_weak   > 0 else weak.iloc[:0]
    train = (
        pd.concat([sampled_strong, sampled_weak])
        .sample(frac=1, random_state=seed)
        .reset_index(drop=True)
    )
    train_original_indices = pd.concat([sampled_strong, sampled_weak]).index
    test = df.drop(index=train_original_indices).reset_index(drop=True)
    if TEST_SIZE is not None:
        test = test.sample(n=min(TEST_SIZE, len(test)), random_state=seed).reset_index(drop=True)
    return train, test

# ==============================================================================
# PHASE A — PAIR ANALYSIS + PER-PAIR NIELSEN REVISION HINTS
# For each training pair we run TWO LLM calls in parallel:
#   (a) original feature extraction (unchanged)
#   (b) per-pair Nielsen revision suggestion
# ==============================================================================
def phase_a(train_df: pd.DataFrame, annotator_id: str) -> list:
    print(f"\n  --- Phase A: Pair Analysis + Nielsen Hints ({len(train_df)} pairs / {MAX_WORKERS} workers) ---")
    analysis_results = []

    nielsen_ids_block = "\n".join(
        f"  {h['id']} ({h['criterion']}): {h['description']}"
        for h in NIELSENS_ORIGINAL
    )

    def _worker(idx, row):
        cid      = row.get("cid", "unknown")
        img_a_b64 = encode_image(os.path.join(IMAGES_DIR, row["left_file"]))
        img_b_b64 = encode_image(os.path.join(IMAGES_DIR, row["right_file"]))
        if not img_a_b64 or not img_b_b64:
            return {"skip": True, "reason": "missing image"}

        winner              = get_winner_label(row)
        loser               = "B" if winner == "A" else "A"
        preference_strength = get_preference_strength(row)
        final_choice_raw    = str(row.get("final_choice", "")).strip()
        intensity_note = (
            "This was a STRONG preference (the user was very decisive)."
            if preference_strength == 2
            else "This was a MILD preference (the user had a slight lean)."
        )

        # --- Call (a): feature extraction (original prompt, unchanged) ---
        ANALYSIS_PROMPT = f"""You are an expert UX design researcher discovering a user's NICHE taste.
The user looked at two UI designs: Design A and Design B.
The user's raw choice label was: "{final_choice_raw}"
The user EXPLICITLY CHOSE: Design {winner}. {intensity_note}

Look at both designs carefully.

1. Identify up to 7 design dimensions where they differ.
2. For each dimension, state the trait in the CHOSEN design ({winner}) vs the REJECTED design ({loser}).
3. Also write a "why_user_rejected_loser" summary.
4. For each feature, assign confidence: "high", "medium", or "low".

Return ONLY a JSON object:
{{
  "why_user_chose_winner": "<max 15 words>",
  "why_user_rejected_loser": "<max 15 words>",
  "preferred_features": [
    {{
      "dimension": "<snake_case_name>",
      "preferred_trait_in_winner": "<brief trait in Design {winner}>",
      "rejected_trait_in_loser": "<brief trait in Design {loser}>",
      "confidence": "<high|medium|low>"
    }}
  ]
}}"""

        # --- Call (b): per-pair Nielsen revision hint ---
        NIELSEN_HINT_PROMPT = f"""You are a UX evaluator adapting Nielsen's 10 Usability Heuristics to match a specific user's preferences.

The user looked at two UI designs: Design A and Design B.
The user CHOSE: Design {winner}. {intensity_note}

Nielsen's 10 Heuristics (current):
{nielsen_ids_block}

Examine both designs.
Based on WHICH heuristics best explain why the user chose Design {winner} over Design {loser}:

1. For each of the 10 heuristics, suggest whether its weight should INCREASE, DECREASE, or STAY THE SAME for this user.
2. If the standard description fails to capture what this user actually cares about, suggest a revised description.
   Otherwise keep the original description.
3. Only mark a heuristic as truly "predictive" if you can point to a clear visual difference between A and B on that dimension.

Return ONLY a JSON object:
{{
  "pair_explanation": "<max 20 words: why the user chose {winner}>",
  "heuristic_hints": [
    {{
      "id": "<H1..H10>",
      "weight_direction": "<increase|decrease|unchanged>",
      "revised_description": "<keep original text if no change needed, else rewrite concisely>",
      "predictive_for_this_pair": <true|false>,
      "confidence": "<high|medium|low>"
    }}
  ]
}}"""

        try:
            # Run both LLM calls concurrently within the worker
            with concurrent.futures.ThreadPoolExecutor(max_workers=2) as inner:
                fut_a = inner.submit(call_llm, ANALYSIS_PROMPT, img_a_b64, img_b_b64)
                fut_b = inner.submit(call_llm, NIELSEN_HINT_PROMPT, img_a_b64, img_b_b64)
                feature_result = fut_a.result()
                nielsen_hint   = fut_b.result()

            feature_valid = isinstance(feature_result, dict) and isinstance(
                feature_result.get("preferred_features"), list
            )
            hint_valid = isinstance(nielsen_hint, dict) and isinstance(
                nielsen_hint.get("heuristic_hints"), list
            )

            return {
                "skip": False,
                "pair_id": row.get("pair_id"),
                "cid": cid,
                "human_choice": winner,
                "preference_strength": preference_strength,
                "final_choice_raw": final_choice_raw,
                "llm_output_valid": feature_valid,
                "llm_analysis": feature_result,
                "nielsen_hint_valid": hint_valid,
                "nielsen_hint": nielsen_hint,
            }
        except Exception as e:
            return {
                "skip": False,
                "pair_id": row.get("pair_id"),
                "cid": cid,
                "human_choice": winner,
                "preference_strength": preference_strength,
                "final_choice_raw": final_choice_raw,
                "llm_output_valid": False,
                "llm_analysis": None,
                "nielsen_hint_valid": False,
                "nielsen_hint": None,
                "error": str(e),
            }

    with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(_worker, idx, row): (idx, row) for idx, row in train_df.iterrows()}
        for future in concurrent.futures.as_completed(futures):
            idx, row = futures[future]
            pair_num = idx + 1
            cid = row.get("cid", "unknown")
            try:
                res = future.result()
            except Exception as e:
                print(f"  -> Fatal Thread Error on {pair_num}: {e}")
                continue
            if res.get("skip"):
                print(f"  -> Skipping {pair_num} ({res.get('reason')})")
                continue
            clean_res = {k: v for k, v in res.items() if k != "skip"}
            analysis_results.append(clean_res)
            if "error" in res:
                print(f"  Analyzed {pair_num}/{len(train_df)} [cid={cid}] -> Failed: {res['error']}")
            else:
                print(
                    f"  Analyzed {pair_num}/{len(train_df)} [cid={cid}] "
                    f"[choice={res['final_choice_raw']}] [winner={res['human_choice']}] "
                    f"[strength={res['preference_strength']}] "
                    f"feature_valid={res['llm_output_valid']} hint_valid={res['nielsen_hint_valid']}"
                )
    return analysis_results


# ==============================================================================
# PHASE C — AGGREGATE REVISION HINTS → REVISED NIELSEN'S
# Takes all per-pair hints, aggregates weight_direction votes (weighted by
# preference_strength), then asks the LLM to produce the final revised heuristics.
# ==============================================================================
def phase_c_nielsen(analysis_results: list) -> list:
    """
    Returns `revised_nielsens`: same structure as NIELSENS_ORIGINAL but with
    updated descriptions and weights.
    """
    print(f"\n  --- Phase C: Aggregating Nielsen Hints ({len(analysis_results)} pairs) ---")

    # Collect hints per heuristic ID
    hint_pool: Dict[str, list] = {h["id"]: [] for h in NIELSENS_ORIGINAL}
    for a in analysis_results:
        strength = int(a.get("preference_strength", 1) or 1)
        hints = (a.get("nielsen_hint") or {}).get("heuristic_hints", [])
        if not hints:
            continue
        for h in hints:
            hid = h.get("id")
            if hid not in hint_pool:
                continue
            if h.get("confidence", "medium") == "low":
                continue  # drop noisy hints
            hint_pool[hid].append({
                "weight_direction": h.get("weight_direction", "unchanged"),
                "revised_description": h.get("revised_description", ""),
                "predictive": bool(h.get("predictive_for_this_pair", False)),
                "evidence_weight": strength,
            })

    # Compute preliminary numeric weight adjustments via vote counting
    preliminary = []
    for h in NIELSENS_ORIGINAL:
        hid   = h["id"]
        pool  = hint_pool[hid]
        base  = h["weight"]
        if not pool:
            preliminary.append({**h, "vote_summary": "no hints", "adjusted_weight": base})
            continue

        score = 0.0
        for hint in pool:
            ew = hint["evidence_weight"]
            if hint["weight_direction"] == "increase":
                score += ew
            elif hint["weight_direction"] == "decrease":
                score -= ew
            # unchanged → 0

        total_evidence = sum(p["evidence_weight"] for p in pool)
        # Normalise to ±1 scale, then shift base weight
        normalised_delta = (score / total_evidence) if total_evidence > 0 else 0.0
        adjusted_weight  = round(max(0.1, base + normalised_delta), 3)

        # Collect candidate revised descriptions (from predictive, high-evidence hints)
        candidate_descs = [
            p["revised_description"]
            for p in pool
            if p["predictive"] and p["revised_description"].strip()
        ]

        preliminary.append({
            "id": hid,
            "criterion": h["criterion"],
            "original_description": h["description"],
            "original_weight": base,
            "adjusted_weight": adjusted_weight,
            "candidate_revised_descriptions": candidate_descs[:5],  # top 5 candidates
            "vote_summary": f"score={score:.1f} over {len(pool)} hints",
        })

    # Now ask the LLM to synthesise the final revised Nielsen's
    SYNTHESIS_PROMPT = f"""You are a UX researcher adapting Nielsen's 10 Usability Heuristics to fit a specific user's preferences.

Below is a preliminary analysis of how each heuristic should change, derived from {len(analysis_results)} training pairs where a human annotator's choices were observed.

Preliminary heuristic adjustments:
{json.dumps(preliminary, indent=2)}

For EACH of the 10 heuristics produce a FINAL revised version:
1. Set a final `weight` (float). Use the `adjusted_weight` as a strong prior, but consider whether candidate descriptions
   suggest the heuristic is highly relevant (push weight up) or irrelevant (push weight down) for this user.
2. Write a `revised_description`: if candidate descriptions reveal a consistent pattern, synthesise them into one
   concise sentence that captures what THIS USER actually cares about on this dimension. If there is no consistent
   signal, keep the original description verbatim.
3. Write `change_rationale`: ≤15 words explaining what changed (or "No change").

Return ONLY a JSON array of 10 objects:
[
  {{
    "id": "H1",
    "criterion": "<original snake_case>",
    "revised_description": "<final description>",
    "weight": <float>,
    "change_rationale": "<≤15 words>"
  }},
  ...
]"""

    print("  Synthesising revised Nielsen's heuristics...")
    try:
        raw = call_llm(SYNTHESIS_PROMPT, temperature=0.2)
        # call_llm returns a dict; if synthesis returns a list it comes back as the root
        if isinstance(raw, list):
            revised_list = raw
        elif isinstance(raw, dict):
            # Sometimes the model wraps the array under a key
            revised_list = next(
                (v for v in raw.values() if isinstance(v, list)), None
            )
            if revised_list is None:
                raise ValueError(f"Unexpected synthesis response shape: {list(raw.keys())}")
        else:
            raise ValueError(f"Unexpected synthesis response type: {type(raw)}")

        # Validate and fill missing fields
        revised_nielsens = []
        original_by_id   = {h["id"]: h for h in NIELSENS_ORIGINAL}
        for item in revised_list:
            hid = item.get("id", "")
            orig = original_by_id.get(hid, {})
            revised_nielsens.append({
                "id": hid,
                "criterion": item.get("criterion", orig.get("criterion", hid)),
                "revised_description": item.get("revised_description", orig.get("description", "")),
                "original_description": orig.get("description", ""),
                "weight": float(item.get("weight", orig.get("weight", 1.0))),
                "original_weight": orig.get("weight", 1.0),
                "change_rationale": item.get("change_rationale", "No change"),
            })

        print(f"  Revised Nielsen's ({len(revised_nielsens)} heuristics):")
        for h in revised_nielsens:
            changed = "✓" if h["revised_description"] != h["original_description"] or abs(h["weight"] - h["original_weight"]) > 0.05 else " "
            print(f"    [{changed}] {h['id']} w={h['weight']:.2f}  {h['change_rationale']}")

        return revised_nielsens

    except Exception as e:
        print(f"  -> Synthesis failed: {e}. Falling back to adjusted weights only.")
        # Fallback: use preliminary adjusted weights, keep original descriptions
        return [
            {
                "id": p["id"],
                "criterion": p["criterion"],
                "revised_description": p["original_description"],
                "original_description": p["original_description"],
                "weight": p["adjusted_weight"],
                "original_weight": p["original_weight"],
                "change_rationale": p.get("vote_summary", "fallback"),
            }
            for p in preliminary
        ]


# ==============================================================================
# SHARED PHASE D SCORER
# Reused for both the Nielsen's baseline run and the revised run.
# `heuristics` is either NIELSENS_ORIGINAL or revised_nielsens.
# ==============================================================================
def _score_pair_once(
    img_a_b64: str,
    img_b_b64: str,
    heuristics: list,
    profile_note: str,
    temperature: float = 0.3,
) -> dict:
    """
    Score a pair against the provided heuristics list.
    `profile_note` is a short string injected into the prompt to distinguish
    the baseline vs revised run in the LLM's framing.
    """
    criteria_json = json.dumps(
        [{"id": h["id"], "criterion": h["criterion"], "description": h.get("revised_description") or h.get("description")} for h in heuristics],
        indent=2
    )
    weights_map = {h["criterion"]: float(h.get("weight", 1.0)) for h in heuristics}

    EVAL_PROMPT = f"""You are scoring two UI designs (A and B) against usability heuristics.
{profile_note}

Heuristics to apply:
{criteria_json}

For EACH heuristic, assign a RELATIVE score: how much better does Design A satisfy this heuristic
compared to Design B, from -5 to +5.
  Positive = A is better.  Negative = B is better.  0 = genuinely equal.
Use the full range. Add a confidence field: "high", "medium", or "low".

Return ONLY a JSON object:
{{
  "criteria_evaluations": [
    {{
      "criterion": "<must match criterion field>",
      "relative_score_a_minus_b": <integer -5 to +5>,
      "reason": "<max 7 words>",
      "confidence": "<high|medium|low>"
    }}
  ]
}}"""

    result = call_llm(EVAL_PROMPT, img_a_b64, img_b_b64, temperature=temperature)

    if not (isinstance(result, dict) and isinstance(result.get("criteria_evaluations"), list)):
        return {"valid": False, "total_delta": 0.0, "evaluations": []}

    total_delta = 0.0
    evaluations = []
    for eval_item in result.get("criteria_evaluations", []):
        crit_name  = eval_item.get("criterion", "")
        confidence = eval_item.get("confidence", "medium")
        try:
            delta = float(eval_item.get("relative_score_a_minus_b", 0))
        except (ValueError, TypeError):
            delta = 0.0

        effective_delta = 0.0 if (confidence == "low" or abs(delta) < NON_DISCRIMINATING_MARGIN) else delta
        weight          = weights_map.get(crit_name, 1.0)
        weighted_delta  = effective_delta * weight

        evaluations.append({
            **eval_item,
            "effective_delta":  effective_delta,
            "applied_weight":   weight,
            "weighted_delta":   weighted_delta,
        })
        total_delta += weighted_delta

    return {"valid": True, "total_delta": total_delta, "evaluations": evaluations}


def _majority_vote_predict(
    img_a_b64: str,
    img_b_b64: str,
    heuristics: list,
    profile_note: str,
) -> dict:
    """Run PHASE_D_VOTE_RUNS scoring calls and return aggregated result."""
    run_results = [
        _score_pair_once(img_a_b64, img_b_b64, heuristics, profile_note, temperature=0.3)
        for _ in range(PHASE_D_VOTE_RUNS)
    ]
    valid_runs = [r for r in run_results if r["valid"]]
    if not valid_runs:
        return {"valid": False, "avg_delta": 0.0, "predicted_choice": None, "votes_a": 0, "votes_b": 0, "votes_tie": 0, "best_evaluations": []}

    avg_delta = sum(r["total_delta"] for r in valid_runs) / len(valid_runs)
    votes_a   = sum(1 for r in valid_runs if r["total_delta"] > 0)
    votes_b   = sum(1 for r in valid_runs if r["total_delta"] < 0)
    votes_tie = len(valid_runs) - votes_a - votes_b

    if abs(avg_delta) < TIE_MARGIN:
        predicted_choice = "A" if votes_a >= votes_b else "B"
    else:
        predicted_choice = "A" if avg_delta > 0 else "B"

    best_run = max(valid_runs, key=lambda r: abs(r["total_delta"]))
    return {
        "valid": True,
        "avg_delta": avg_delta,
        "predicted_choice": predicted_choice,
        "votes_a": votes_a,
        "votes_b": votes_b,
        "votes_tie": votes_tie,
        "best_evaluations": best_run["evaluations"],
    }


def phase_d_dual(
    test_df: pd.DataFrame,
    revised_nielsens: list,
) -> tuple[list, dict, dict]:
    """
    Runs BOTH baseline (original Nielsen's) and revised Nielsen's evaluations
    on every test pair — in parallel per pair, sequentially across baseline/revised
    within each worker to avoid doubling API concurrency.

    Returns:
        predictions      : list of per-pair dicts with both baseline and revised results
        baseline_metrics : {"accuracy", "correct", "total"}
        revised_metrics  : {"accuracy", "correct", "total"}
    """
    print(
        f"\n  --- Phase D (Dual): {len(test_df)} pairs / {MAX_WORKERS} workers / "
        f"{PHASE_D_VOTE_RUNS} votes each × 2 runs (baseline + revised) ---"
    )

    BASELINE_NOTE = "Apply the standard Nielsen's Usability Heuristics as written."
    REVISED_NOTE  = "Apply these heuristics which have been adapted to match this specific user's preferences."

    def _worker(idx, row):
        cid      = row.get("cid", "unknown")
        img_a_b64 = encode_image(os.path.join(IMAGES_DIR, row["left_file"]))
        img_b_b64 = encode_image(os.path.join(IMAGES_DIR, row["right_file"]))
        if not img_a_b64 or not img_b_b64:
            return {"skip": True, "reason": "missing image"}

        true_winner      = get_winner_label(row)
        final_choice_raw = str(row.get("final_choice", "")).strip()
        true_strength    = get_preference_strength(row)

        try:
            # Run baseline and revised concurrently within the worker
            with concurrent.futures.ThreadPoolExecutor(max_workers=2) as inner:
                fut_base = inner.submit(
                    _majority_vote_predict, img_a_b64, img_b_b64, NIELSENS_ORIGINAL, BASELINE_NOTE
                )
                fut_rev = inner.submit(
                    _majority_vote_predict, img_a_b64, img_b_b64, revised_nielsens, REVISED_NOTE
                )
                base_result = fut_base.result()
                rev_result  = fut_rev.result()

            base_pred    = base_result.get("predicted_choice")
            rev_pred     = rev_result.get("predicted_choice")
            base_correct = base_pred == true_winner if base_pred else False
            rev_correct  = rev_pred  == true_winner if rev_pred  else False

            return {
                "skip": False,
                "pair_id": row.get("pair_id"),
                "cid": cid,
                "true_winner": true_winner,
                "final_choice_raw": final_choice_raw,
                "true_strength": true_strength,
                # Baseline
                "baseline_predicted": base_pred,
                "baseline_correct": base_correct,
                "baseline_valid": base_result["valid"],
                "baseline_avg_delta": base_result.get("avg_delta", 0.0),
                "baseline_votes": {"a": base_result.get("votes_a"), "b": base_result.get("votes_b"), "tie": base_result.get("votes_tie")},
                "baseline_evaluations": base_result.get("best_evaluations", []),
                # Revised
                "revised_predicted": rev_pred,
                "revised_correct": rev_correct,
                "revised_valid": rev_result["valid"],
                "revised_avg_delta": rev_result.get("avg_delta", 0.0),
                "revised_votes": {"a": rev_result.get("votes_a"), "b": rev_result.get("votes_b"), "tie": rev_result.get("votes_tie")},
                "revised_evaluations": rev_result.get("best_evaluations", []),
            }
        except Exception as e:
            return {
                "skip": False,
                "pair_id": row.get("pair_id"),
                "cid": cid,
                "true_winner": true_winner,
                "final_choice_raw": final_choice_raw,
                "true_strength": true_strength,
                "baseline_predicted": None, "baseline_correct": False, "baseline_valid": False,
                "revised_predicted":  None, "revised_correct":  False, "revised_valid":  False,
                "error": str(e),
            }

    predictions = []
    base_correct_total = base_valid_total = 0
    rev_correct_total  = rev_valid_total  = 0

    with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(_worker, idx, row): (idx, row) for idx, row in test_df.iterrows()}
        for future in concurrent.futures.as_completed(futures):
            idx, row = futures[future]
            pair_num = idx + 1
            cid = row.get("cid", "unknown")
            try:
                res = future.result()
            except Exception as e:
                print(f"  -> Fatal Thread Error on {pair_num}: {e}")
                continue
            if res.get("skip"):
                print(f"  -> Skipping {pair_num} ({res['reason']})")
                continue

            clean_res = {k: v for k, v in res.items() if k != "skip"}
            predictions.append(clean_res)

            if res.get("baseline_valid"):
                base_valid_total += 1
                if res["baseline_correct"]:
                    base_correct_total += 1
            if res.get("revised_valid"):
                rev_valid_total += 1
                if res["revised_correct"]:
                    rev_correct_total += 1

            base_acc = (base_correct_total / base_valid_total) if base_valid_total else 0.0
            rev_acc  = (rev_correct_total  / rev_valid_total)  if rev_valid_total  else 0.0

            if "error" in res:
                print(f"  Eval {pair_num}/{len(test_df)} [cid={cid}] -> Error: {res['error']}")
            else:
                verdict_base = "✓" if res["baseline_correct"] else "✗"
                verdict_rev  = "✓" if res["revised_correct"]  else "✗"
                print(
                    f"  Eval {pair_num}/{len(test_df)} [cid={cid}] "
                    f"True={res['true_winner']} | "
                    f"Base={res['baseline_predicted']}{verdict_base}(Δ={res['baseline_avg_delta']:.1f}) | "
                    f"Rev={res['revised_predicted']}{verdict_rev}(Δ={res['revised_avg_delta']:.1f}) | "
                    f"Acc base={base_correct_total}/{base_valid_total}({base_acc*100:.1f}%) "
                    f"rev={rev_correct_total}/{rev_valid_total}({rev_acc*100:.1f}%)"
                )

    baseline_metrics = {
        "accuracy": (base_correct_total / base_valid_total) if base_valid_total else 0.0,
        "correct":  base_correct_total,
        "total":    base_valid_total,
    }
    revised_metrics = {
        "accuracy": (rev_correct_total / rev_valid_total) if rev_valid_total else 0.0,
        "correct":  rev_correct_total,
        "total":    rev_valid_total,
    }

    print(f"\n  === ACCURACY COMPARISON ===")
    print(f"  Nielsen's Baseline : {base_correct_total}/{base_valid_total} ({baseline_metrics['accuracy']*100:.2f}%)")
    print(f"  Revised Nielsen's  : {rev_correct_total}/{rev_valid_total}  ({revised_metrics['accuracy']*100:.2f}%)")
    delta = revised_metrics["accuracy"] - baseline_metrics["accuracy"]
    sign  = "+" if delta >= 0 else ""
    print(f"  Delta              : {sign}{delta*100:.2f}%")

    return predictions, baseline_metrics, revised_metrics


# ==============================================================================
# MAIN
# ==============================================================================
def run_pipeline_for_annotator(csv_path: str, annotator_id: str):
    print(f"\n{'='*60}")
    print(f"Processing annotator: {annotator_id}")
    print(f"{'='*60}")
    set_seed(SEED)

    df = load_annotator_data(csv_path)
    print(f"  Loaded {len(df)} rows from {csv_path}")

    if len(df) <= TRAIN_SIZE:
        print(f"  Warning: only {len(df)} rows — need > {TRAIN_SIZE} for a non-empty test set.")

    train_df, test_df = sample_and_split(df, seed=SEED)
    print(f"  Train: {len(train_df)} | Test: {len(test_df)}")

    # Phase A — feature extraction + per-pair Nielsen hints
    analysis_results = phase_a(train_df, annotator_id)

    # Phase C — aggregate hints → revised Nielsen's
    revised_nielsens = phase_c_nielsen(analysis_results)

    # Phase D (dual) — baseline Nielsen's vs revised Nielsen's on test set
    predictions, baseline_metrics, revised_metrics = phase_d_dual(test_df, revised_nielsens)

    # Build heuristic diff table for the report
    heuristic_diff = []
    original_by_id = {h["id"]: h for h in NIELSENS_ORIGINAL}
    for h in revised_nielsens:
        orig = original_by_id.get(h["id"], {})
        heuristic_diff.append({
            "id": h["id"],
            "criterion": h["criterion"],
            "original_description": orig.get("description", ""),
            "revised_description": h.get("revised_description", ""),
            "description_changed": h.get("revised_description", "") != orig.get("description", ""),
            "original_weight": orig.get("weight", 1.0),
            "revised_weight": h.get("weight", 1.0),
            "weight_delta": round(h.get("weight", 1.0) - orig.get("weight", 1.0), 3),
            "change_rationale": h.get("change_rationale", "No change"),
        })

    output = {
        "annotator_id": annotator_id,
        "model": MODEL,
        "sample_sizes": {
            "total_rows": len(df),
            "train": len(train_df),
            "test": len(test_df),
        },
        # Accuracy comparison — the headline result
        "accuracy_comparison": {
            "nielsen_baseline": baseline_metrics,
            "revised_nielsen":  revised_metrics,
            "accuracy_delta":   round(revised_metrics["accuracy"] - baseline_metrics["accuracy"], 4),
        },
        # Revised heuristics with diffs
        "heuristic_diff": heuristic_diff,
        "revised_nielsens": revised_nielsens,
        # Training data
        "pair_analyses": analysis_results,
        # Per-pair test predictions
        "predictions": predictions,
    }

    os.makedirs(RESULTS_DIR, exist_ok=True)
    out_path = os.path.join(RESULTS_DIR, f"{annotator_id}.json")
    with open(out_path, "w") as f:
        json.dump(output, f, indent=2)
    print(f"\n  Saved → {out_path}")
    return output


def main():
    annotator_dir = Path(ANNOTATOR_DIR)
    if not annotator_dir.exists():
        print(f"Error: {ANNOTATOR_DIR} not found.")
        return
    csv_files = sorted(annotator_dir.glob("*.csv"))
    if not csv_files:
        print(f"No CSV files found in {ANNOTATOR_DIR}")
        return
    print(f"Found {len(csv_files)} annotator(s): {[f.stem for f in csv_files]}")
    all_summaries = []
    for csv_path in csv_files:
        annotator_id = csv_path.stem
        try:
            result = run_pipeline_for_annotator(str(csv_path), annotator_id)
            all_summaries.append({
                "annotator_id": annotator_id,
                **result["accuracy_comparison"],
            })
        except Exception as e:
            print(f"\n[ERROR] Failed for annotator {annotator_id}: {e}")

    # Print cross-annotator summary
    if all_summaries:
        print(f"\n{'='*60}")
        print("CROSS-ANNOTATOR SUMMARY")
        print(f"{'='*60}")
        print(f"  {'Annotator':<20} {'Baseline':>10} {'Revised':>10} {'Delta':>10}")
        print(f"  {'-'*50}")
        for s in all_summaries:
            b = s["nielsen_baseline"]["accuracy"] * 100
            r = s["revised_nielsen"]["accuracy"]  * 100
            d = s["accuracy_delta"] * 100
            sign = "+" if d >= 0 else ""
            print(f"  {s['annotator_id']:<20} {b:>9.2f}%  {r:>9.2f}%  {sign}{d:>8.2f}%")

    print("\nAll annotators processed.")


if __name__ == "__main__":
    main()

Found 20 annotator(s): ['0c65ba0b46894372', '1d3ee9b46ac34e6c', '247b7dfa8a5347ad', '2c79548f2ab44243', '2d8219ac74d146ba', '454c297fa1e54296', '498b9ea72d994e8e', '653f05d5c9ab4c97', '6ccbe484cb96425f', '7602a5d37b7d4220', '7a945bb87f2b4cd2', '8287fcc5b6504e39', '8f61f5a0685f42ec', '97945b44a2b54914', 'a692cdb11bbe432c', 'abda3f52c24c4038', 'ad7def63e86045b1', 'dad4b876ad3147e4', 'e8f37a526c444958', 'eee1fd24aad648ed']

Processing annotator: 0c65ba0b46894372
  Loaded 610 rows from annotator/0c65ba0b46894372.csv
  Train: 30 | Test: 30

  --- Phase A: Pair Analysis + Nielsen Hints (30 pairs / 7 workers) ---
  Analyzed 4/30 [cid=13585::gemini_comp_s2w_13585_v2.png>>gemini_comp_s2w_13585_v3.png::T000021] [choice=A < B] [winner=B] [strength=1] feature_valid=True hint_valid=True
  Analyzed 1/30 [cid=68037::gemini_comp_s2w_68037_v3.png>>gemini_comp_s2w_68037_v4.png::T000497] [choice=A << B] [winner=B] [strength=2] feature_valid=True hint_valid=True
  Analyzed 5/30 [cid=68136::gemini_comp_s2w

KeyboardInterrupt: 